In [2]:
import os
from pathlib import Path

os.environ["HADOOP_HOME"] = "C:\\Users\\danye\\Documents\\hadoop"
os.environ["PATH"] = os.environ["HADOOP_HOME"] + "\\bin;" + os.environ["PATH"]

import numpy as np
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

RUTA_BASE = Path("..")
RUTA_DATOS = RUTA_BASE / "Datos"
RUTA_FEATURES = RUTA_DATOS / "Analiticos" / "Parquet" / "dataset_guia03_features.parquet"

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("ProyectoFinal_Guia08")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

print("Spark:", spark.version)

Spark: 4.2.0


In [3]:
df = spark.read.parquet(str(RUTA_FEATURES.resolve()))
total = df.count()
print("Filas:", total, "| Columnas:", len(df.columns))

df = df.withColumn("es_ataque", F.when(F.col("label") == "BENIGN", 0).otherwise(1))

print("\nDistribución del target:")
(
    df.groupBy("es_ataque").count()
    .withColumn("porcentaje", F.round(F.col("count") / total * 100, 2))
    .show()
)

print("Detalle por tipo de tráfico:")
df.groupBy("label").count().orderBy(F.desc("count")).show(20, truncate=False)

Filas: 2827562 | Columnas: 87

Distribución del target:
+---------+-------+----------+
|es_ataque|  count|porcentaje|
+---------+-------+----------+
|        1| 556555|     19.68|
|        0|2271007|     80.32|
+---------+-------+----------+

Detalle por tipo de tráfico:
+--------------------------+-------+
|label                     |count  |
+--------------------------+-------+
|BENIGN                    |2271007|
|DoS Hulk                  |230123 |
|PortScan                  |158804 |
|DDoS                      |128025 |
|DoS GoldenEye             |10293  |
|FTP-Patator               |7935   |
|SSH-Patator               |5897   |
|DoS slowloris             |5796   |
|DoS Slowhttptest          |5499   |
|Bot                       |1956   |
|Web Attack - Brute Force  |1507   |
|Web Attack - XSS          |652    |
|Infiltration              |36     |
|Web Attack - Sql Injection|21     |
|Heartbleed                |11     |
+--------------------------+-------+



In [4]:
TARGET = "es_ataque"

COLS_NUMERICAS = [
    "flow_duration", "total_fwd_packets", "total_backward_packets",
    "total_length_of_fwd_packets", "total_length_of_bwd_packets",
    "fwd_packet_length_mean", "bwd_packet_length_mean",
    "packet_length_mean", "packet_length_std",
    "flow_bytes_s", "flow_packets_s",
    "flow_iat_mean", "fwd_iat_mean", "bwd_iat_mean",
    "down_up_ratio", "init_win_bytes_forward", "init_win_bytes_backward",
    "ratio_fwd_bwd_bytes", "total_flags_activadas",
]
COLS_CATEGORICAS = ["protocol", "categoria_duracion"]
COLS_MODELO = COLS_NUMERICAS + COLS_CATEGORICAS

print("Columnas que no existen en el dataset:", [c for c in COLS_MODELO if c not in df.columns])

FRACCION_MUESTRA = 0.20
df_muestra = df.sampleBy(TARGET, fractions={0: FRACCION_MUESTRA, 1: FRACCION_MUESTRA}, seed=42)
pdf = df_muestra.select(COLS_MODELO + [TARGET]).toPandas()

print("\nMuestra:", pdf.shape)
print(pdf[TARGET].value_counts(normalize=True).round(4))

nulos = pdf.isnull().sum()
print("\nNulos por columna:")
print(nulos[nulos > 0] if (nulos > 0).any() else "Sin nulos")

infinitos = np.isinf(pdf[COLS_NUMERICAS]).sum()
print("\nInfinitos por columna:")
print(infinitos[infinitos > 0] if (infinitos > 0).any() else "Sin infinitos")

Columnas que no existen en el dataset: []


C:\Users\danye\anaconda3\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()



Muestra: (566097, 22)
es_ataque
0    0.803
1    0.197
Name: proportion, dtype: float64

Nulos por columna:
Sin nulos

Infinitos por columna:
Sin infinitos


In [5]:
X = pdf[COLS_MODELO]
y = pdf[TARGET]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

for nombre, yy in [("Entrenamiento", y_train), ("Validación", y_val), ("Prueba", y_test)]:
    print(f"{nombre}: {len(yy)} registros | % ataque: {yy.mean() * 100:.2f}")

Entrenamiento: 396267 registros | % ataque: 19.70
Validación: 84915 registros | % ataque: 19.70
Prueba: 84915 registros | % ataque: 19.70


In [6]:
preprocesador = ColumnTransformer([
    ("num", StandardScaler(), COLS_NUMERICAS),
    ("cat", OneHotEncoder(handle_unknown="ignore"), COLS_CATEGORICAS),
])

modelo = Pipeline([
    ("prep", preprocesador),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
])

modelo.fit(X_train, y_train)

print("Modelo entrenado")
print("Iteraciones usadas:", modelo.named_steps["clf"].n_iter_[0])

Modelo entrenado
Iteraciones usadas: 213


In [7]:
y_pred = modelo.predict(X_test)
y_prob = modelo.predict_proba(X_test)[:, 1]

metricas = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred),
    "F1-Score": f1_score(y_test, y_pred),
    "ROC-AUC": roc_auc_score(y_test, y_prob),
}

print("Métricas en el conjunto de prueba:")
for nombre, valor in metricas.items():
    print(f"{nombre}: {valor:.4f}")

tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
print("\nMatriz de confusión:")
print(f"TN = {tn} | FP = {fp}")
print(f"FN = {fn} | TP = {tp}")

Métricas en el conjunto de prueba:
Accuracy: 0.8457
Precision: 0.5644
Recall: 0.9503
F1-Score: 0.7082
ROC-AUC: 0.9558

Matriz de confusión:
TN = 55922 | FP = 12268
FN = 831 | TP = 15894


In [8]:
nombres = modelo.named_steps["prep"].get_feature_names_out()
coefs = modelo.named_steps["clf"].coef_[0]

df_coef = pd.DataFrame({"variable": nombres, "coeficiente": coefs})
df_coef["variable"] = df_coef["variable"].str.replace("num__", "").str.replace("cat__", "")
df_coef["abs"] = df_coef["coeficiente"].abs()
df_coef["importancia"] = pd.qcut(
    df_coef["abs"].rank(method="first"), 3, labels=["Baja", "Media", "Alta"]
)

df_coef = df_coef.sort_values("abs", ascending=False).reset_index(drop=True)
print(df_coef[["variable", "coeficiente", "importancia"]].round(4).to_string())

                       variable  coeficiente importancia
0        total_backward_packets     -37.0367        Alta
1             total_fwd_packets     -35.3724        Alta
2                   protocol_17     -13.6947        Alta
3   total_length_of_bwd_packets     -13.0730        Alta
4                    protocol_6      12.9122        Alta
5        fwd_packet_length_mean      -2.5095        Alta
6         total_flags_activadas      -2.4670        Alta
7                  fwd_iat_mean       1.5197        Alta
8                    protocol_0       1.1943       Media
9             packet_length_std       0.9130       Media
10      init_win_bytes_backward      -0.7331       Media
11     categoria_duracion_Corto       0.6574       Media
12          ratio_fwd_bwd_bytes       0.5903       Media
13  total_length_of_fwd_packets       0.5800       Media
14       bwd_packet_length_mean       0.5113       Media
15                flow_duration      -0.5065       Media
16                down_up_ratio